In [ ]:
import cobra
import pandas as pd

from Bio import SeqIO
from copy import deepcopy

from utils.cfr_suite import apply_cfr 
from HV_utils.genVBOF2 import genVBOF2, genHVM
from HV_utils.info import metDict, ntpsDict

In [4]:
model = cobra.io.read_sbml_model("gems/Recon2.2.xml")
hvmodel = deepcopy(model)

         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/88/pb4s0b9n0s9g7yzmv_sft6pr0000gn/T/tmpt0t02e7u.lp
Reading time = 0.02 seconds
: 6047 rows, 17016 columns, 67648 nonzeros


In [5]:
virus_record = SeqIO.read("./data/NC_001357.gb", "genbank")
virus_record

SeqRecord(seq=Seq('ATTAATACTTTTAACAATTGTAGTATATAAAAAAGGGAGTAACCGAAAACGGTC...TTC'), id='NC_001357.1', name='NC_001357', description='Human papillomavirus - 18, complete genome', dbxrefs=['BioProject:PRJNA485481'])

In [6]:
vbof = genVBOF2(virus_record, hvmodel)

In [7]:
vbof

Reaction identifier,HpV18_prodrxn_VN
Name,Human papillomavirus - 18 production reaction
Memory address,0x32d89aa50
Stoichiometry,0.3546927817891488 ala_L_c + 0.32063042696179844 arg_L_c + 0.2473223154855443 asn_L_c + 0.39838145428509825 asp_L_c + 27.815505365666542 atp_c + 0.0006983589679450848 ctp_c + 0.17845711985633583... 0.3546927817891488 L-alanine + 0.32063042696179844 L-argininium(1+) + 0.2473223154855443 L-asparagine + 0.39838145428509825 L-aspartate(1-) + 27.815505365666542 ATP(4-) + 0.0006983589679450848...
GPR,
Lower bound,0
Upper bound,1000


In [ ]:
# reverse the id strings
met_id_lookup = {v: k for k, v in MET_IDS.items()}

dna_mass = 0
protein_mass = 0
rna_mass = 0
for m, s in sorted(
    model.reactions.PHM2_prodrxn_VN.metabolites.items(), key=lambda x: x[0].id
):
    if s > 0:
        continue  # only looking at consumption right now
    if m.id in ["dATP[c]", "dCTP[c]", "dGTP[c]", "dTTP[c]"]:
        mp = m.id[0:2] + "M" + m.id[3:]
        lookup = met_id_lookup[mp]
        dna_mass -= NUCLEOTIDE_MASS[lookup] * s / 1000
    elif m.id in ["ATP[c]", "CTP[c]", "GTP[c]", "UTP[c]"]:
        lookup = met_id_lookup[m.id]
        if m.id == "ATP[c]":  # for ATP, we separate out energetic and RNA requirements
            sd = model.reactions.PHM2_prodrxn_VN.metabolites[
                model.metabolites.get_by_id("H2O[c]")
            ]
            rna_mass -= NUCLEOTIDE_MASS[lookup] * (s - sd) / 1000
            # other_masses -= NUCLEOTIDE_MASS[lookup] * sd / 1000
        else:
            rna_mass -= NUCLEOTIDE_MASS[lookup] * s / 1000
    elif m.id.startswith("L_") or m.id == "Glycine[c]":
        lookup = met_id_lookup[m.id]
        protein_mass -= AMINO_ACID_MASS[lookup] * s / 1000

print(f"{dna_mass=}, {protein_mass=}, {rna_mass=}")